### 1. Plot diurnal cycles from model data
Load series extracted from blurred data
Groups data by season, then averages by hour of day to give the diurnal cycle.

In [3]:
from Montreal_UHI_toolbox import *
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

# station_set = obs

/runoff/gulley/.miniconda3/lib/python3.12/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.39.0 or higher is recommended. You are running version 2.14.1
  warnings.warn(


In [4]:
series = {}
for urban_or_rural in ['urban','rural']:
    series[urban_or_rural] = {}
    for key in ['tas_C','tas_T']:
        series[urban_or_rural][key] = xr.open_mfdataset(f'/runoff/gulley/St_Laurent/intermediates/sim/series_at_obs_locations/{key}_{urban_or_rural}.nc')

In [32]:
diurnal = {}
diurnal_CI = {}
diurnal_interval = {}
grouped_by_year = {}
n = 23 # number of sampled years
alpha = 0.05 # for CI based on inter-annual variability

for urban_or_rural in ['urban','rural']:
    diurnal[urban_or_rural] = {}
    diurnal_CI[urban_or_rural] = {}
    diurnal_interval[urban_or_rural] = {}
    grouped_by_year[urban_or_rural] = {}

    for key in ['tas_C','tas_T']:
        diurnal[urban_or_rural][key] = {}
        diurnal_CI[urban_or_rural][key] = {}
        diurnal_interval[urban_or_rural][key] = {}
        grouped_by_year[urban_or_rural][key] = {}

        for season in ['JJA','SON','DJF','MAM']:
            selected_by_season = series[urban_or_rural][key].sel(time=series[urban_or_rural][key].time.dt.season == season)
            diurnal[urban_or_rural][key][season] = selected_by_season.groupby('time.hour').mean(dim='time').mean(dim='points')
            # diurnal_CI[urban_or_rural][key][season] = ( selected_by_season.groupby('time.hour').std(dim='time').mean(dim='points')/np.sqrt(n) ) * stats.t.ppf(1 - alpha/2, n-1)
            diurnal_CI[urban_or_rural][key][season] =  (selected_by_season.mean(dim='points').groupby('time.hour').std(dim='time', ddof=1) / np.sqrt(n)) * stats.t.ppf(1 - alpha/2, n-1)
            diurnal_interval[urban_or_rural][key][season] =  selected_by_season.mean(dim='points').groupby(['time.hour','time.year']).mean(dim='time').quantile([0.05,0.95], dim='year')
            
            grouped_by_year[urban_or_rural][key][season] = selected_by_season.groupby('time.year')

years = grouped_by_year['urban']['tas_C']['JJA'].groups.keys()

In [67]:
diurnal_interval[urban_or_rural][key][season].sel(quantile=0.95).tas.values

array([297.28666593, 295.84900949, 294.92451992, 294.11127251,
       293.43383742, 292.87777506, 292.26324936, 291.718739  ,
       291.23756785, 290.81044911, 290.8001984 , 292.14950026,
       293.64874276, 295.06536323, 296.32174936, 297.46469345,
       298.54502236, 299.36358524, 299.99753657, 300.44045047,
       300.55889176, 300.47048246, 300.02056339, 298.93429229])

In [47]:
# Managing different elevations for temperature, temperature adjustments are performed in the final step of any rendering
# Extract blurred orography (effective model elevation) from simulation data at station points
adjustment_set = {}
adjustment_set['urban'] = add_blurred_field_to_stations(static_fields_C['orog'],station_set = obs_rural)
adjustment_set['rural'] = add_blurred_field_to_stations(static_fields_C['orog'],station_set = obs_urban)

# Temperature adjustments can be performed after taking the average elevation difference for each set
Z_b_model = {}
Z_b_model['urban'] = np.mean(adjustment_set['urban']['orog_blurred_std1p5'].values) + 2.
Z_b_model['rural'] = np.mean(adjustment_set['rural']['orog_blurred_std1p5'].values) + 2.
print(f'orog (m) before altitude scaling for each station set: {Z_b_model} to be scaled to {Z_a}m')

orog (m) before altitude scaling for each station set: {'urban': 58.576568175475465, 'rural': 46.045161577295985} to be scaled to 54.5m


In [48]:
# Create confidence intervals for the diurnal UHI based on inter-annual variability
diurnal_UHI_CI = {}
for key in ['tas_C','tas_T']:
    diurnal_UHI_CI[key] = {}
    for season in ['JJA','SON','DJF','MAM']:
        selected_by_season_rural = series['rural'][key].sel(time=series['rural'][key].time.dt.season == season).mean(dim='points')
        selected_by_season_urban = series['urban'][key].sel(time=series['urban'][key].time.dt.season == season).mean(dim='points')
        
        # Adjust temperatures based on model orogoraphy in this case
        selected_by_season_rural['tas'] = selected_by_season_rural['tas'].copy(data=adjust_temp(selected_by_season_rural['tas'].values,z_b  = Z_b_model['rural']).squeeze())
        selected_by_season_urban['tas'] = selected_by_season_urban['tas'].copy(data=adjust_temp(selected_by_season_urban['tas'].values,z_b  = Z_b_model['urban']).squeeze())

        diurnal_UHI_CI[key][season] = ((selected_by_season_urban - selected_by_season_rural).groupby('time.hour').std(dim='time')/np.sqrt(n)) * stats.t.ppf(1 - alpha/2, n-1)        

In [ ]:
# Diurnal temperature cycles, mean and by each year
X = diurnal['urban']['tas_C']['JJA'].hour - 4 # Time of day converted to UTC-4
X = np.concatenate([X,X+24]) # Show two diurnal cycles in the x-axis

for season in ['JJA','SON','DJF','MAM']:
    fig = go.Figure()
    for key,model,colour in zip(['tas_C','tas_T'],['CLASS','TEB+CLASS'],['blue','red']):
        
        # Temperatures
        for urban_or_rural, linestyle in zip(['urban','rural'],['solid','dash']):
            
            Y = diurnal[urban_or_rural][key][season].tas.values # Unadjusted for altitude
            # Adjust temperatures by altitude
            #       - Note: z_a as default ensures it outputs temperatures adjusted  station elevations 
            #               (ie not model orography so it must be done for 'urban' too)
            Y = adjust_temp(T_b = Y,                                       # T_b - temperatures before adjustment
                            z_b = Z_b_model[urban_or_rural])[0] - 273.15   # z_b - initial altitude average for the set
            
            # Duplicate to show two cycles on graph
            Y = np.concatenate([Y,Y]) 
            
            LINE = dict(color=colour, dash=linestyle)
            fig.add_trace(go.Scatter(
                x=X,
                y=Y,
                name=f'{model} {urban_or_rural}',
                line=dict(color=colour, dash=linestyle, width=3),
                legendgroup='means',
                legendgrouptitle_text='Mean Cycles',
                legendrank=1
            ))
     
    # Plot other year diurnal cycles to show inter-annual variability
    for year in years:
        for key, model, colour in zip(
            ['tas_C','tas_T'],
            ['CLASS','TEB+CLASS'],
            ['blue','red']
        ):
            for urban_or_rural, linestyle in zip(
                ['urban','rural'],
                ['solid','dash']
            ):

                diurnal_by_year = (
                    grouped_by_year[urban_or_rural][key][season][year]
                    .groupby('time.hour')
                    .mean(dim='time')
                    .mean(dim='points')
                )

                # Temperatures adjusted for elevation, each year, cycled twice on graph per year
                Y = adjust_temp(T_b = diurnal_by_year.tas.values,                                      
                            z_b = Z_b_model[urban_or_rural])[0] - 273.15   
                Y = np.concatenate([Y,Y])

                fig.add_trace(go.Scatter(
                    x=X,
                    y=Y,
                    name=f'{year} {model} {urban_or_rural}',
                    line=dict(color=colour, dash=linestyle, width=0.2),
                    legendrank=2,
                    showlegend=True
                ))

    fig.update_xaxes(
        range=[0, 24],
        tickmode='array',
        tickvals=list(range(0, 25, 3)),
        ticktext=[f'{h:02d}:00' for h in range(0, 25, 3)]
    )

    fig.update_layout(
        title=f'{season} Diurnal Temperature Cycle',
        xaxis_title='Hour of day (UTC-4)',
        yaxis_title='Temperature (°C)',
        legend=dict(
            font=dict(size=9),
            groupclick='toggleitem',
            tracegroupgap=4
        ),
    )
    fig.show()
    # fig.write_html(f'/runoff/gulley/UHI_plots/cycles/diurnal/{season}_diurnal_simobs.html')

In [72]:
# Diurnal temperature cycles, mean and 95 percentile
X = diurnal['urban']['tas_C']['JJA'].hour - 4 # Time of day converted to UTC-4
X = np.concatenate([X,X+24]) # Show two diurnal cycles in the x-axis

for season in ['JJA','SON','DJF','MAM']:
    fig = go.Figure()
    for key,model,colour in zip(['tas_C','tas_T'],['CLASS','TEB+CLASS'],['blue','red']):
        
        # Temperatures
        for urban_or_rural, linestyle in zip(['urban','rural'],['solid','dash']):
            
            Y = diurnal[urban_or_rural][key][season].tas.values # Unadjusted for altitude
            CI = diurnal_CI[urban_or_rural][key][season].tas.values # Absolute 95% confidence values
            upper = diurnal_interval[urban_or_rural][key][season].sel(quantile = 0.95).tas.values
            lower = diurnal_interval[urban_or_rural][key][season].sel(quantile = 0.05).tas.values
            
            
            # Adjust temperatures by altitude
            #       - Note: z_a as default ensures it outputs temperatures adjusted  station elevations 
            #               (ie not model orography so it must be done for 'urban' too)
            Y = adjust_temp(T_b = Y,                                       # T_b - temperatures before adjustment
                            z_b = Z_b_model[urban_or_rural])[0] - 273.15   # z_b - initial altitude average for the set
            upper = adjust_temp(T_b = upper,                                       # T_b - temperatures before adjustment
                                 z_b = Z_b_model[urban_or_rural])[0] - 273.15
            lower = adjust_temp(T_b = lower,                                       # T_b - temperatures before adjustment
                                 z_b = Z_b_model[urban_or_rural])[0] - 273.15

            # Duplicate to show two cycles on graph
            Y = np.concatenate([Y,Y])
            upper = np.concatenate([upper,upper]) 
            lower = np.concatenate([lower,lower]) 

            CI = np.concatenate([CI,CI])

            LINE = dict(color=colour, dash=linestyle)
            fig.add_trace(go.Scatter(
                x=X,
                y=Y,
                name=f'{model} {urban_or_rural}',
                line=dict(color=colour, dash=linestyle, width=3),
                legendgroup='means',
                legendgrouptitle_text='Mean Cycles',
                legendrank=1
            ))

            # For painting intervals onto graph
            opacity = 0.2
            if urban_or_rural == 'rural':
                opacity = 0.1

            if colour == 'red':
                fill_colour = f'rgba(255, 0, 0, {opacity})' # with limited opacity
            else: # blue
                fill_colour =  f'rgba(0, 0, 255, {opacity})'
            
            if urban_or_rural == 'rural':
                fillpattern = dict(
                    shape='/',          # hatch style
                    size=8,             # spacing between lines
                    solidity=0.2,       # density of the pattern (0–1)
                    fgcolor=fill_colour,     # hatch line color
                )
            else:
                fillpattern = None

            # Lower part of confidence intervals
            fig.add_trace(go.Scatter(
                x=X,
                y=lower,
                mode='lines',
                line=dict(width=0,color=colour),          
                showlegend=False,
                name=f'{model} {urban_or_rural} 5th-95th percentile'
            ))

            # Upper part of confidence intervals
            fig.add_trace(go.Scatter(
                x=X,
                y=upper,
                mode='lines',
                line=dict(width=0,color=colour), 
                fill='tonexty',              # fills to the trace added just before this one
                fillcolor=fill_colour,
                showlegend=True,
                fillpattern=fillpattern,
                name=f'{model} {urban_or_rural} 5th-95th percentile'
            ))
     
    fig.update_xaxes(
        range=[0, 24],
        tickmode='array',
        tickvals=list(range(0, 25, 3)),
        ticktext=[f'{h:02d}:00' for h in range(0, 25, 3)]
    )

    fig.update_layout(
        title=f'{season} Diurnal Temperature Cycle',
        xaxis_title='Hour of day (UTC-4)',
        yaxis_title='Temperature (°C)',
        legend=dict(
            font=dict(size=9),
            groupclick='toggleitem',
            tracegroupgap=4
        ),
    )
    # fig.show()
    fig.write_html(f'/runoff/gulley/UHI_plots/cycles/diurnal/{season}_diurnal_simobs_perc.html')

In [131]:
# Display UHI average and by year

X = diurnal['urban']['tas_C']['JJA'].hour - 4 # Time of day converted to UTC-4
X = np.concatenate([X,X+24]) # Show two diurnal cycles in the x-axis

y_range = [-1,3.2] # range of the graph
offset = 0.4       # for annotations to not interfere with axes

for season in ['JJA','SON','DJF','MAM']:
    fig = go.Figure()

    # To connect with hottest/coldest part of the day by season/model/land cover
    i_max_urban = {} 
    i_min_urban = {}
    i_max_rural = {}
    i_min_rural = {}

    for key,model,colour in zip(['tas_C','tas_T'],['CLASS','TEB+CLASS'],['blue','red']):
        
        # UHI
        Y = adjust_temp( # urban adjusted temperatures
            T_b=diurnal['urban'][key][season].tas.values,
            z_b=Z_b_model['urban']
        )[0] - adjust_temp( # rural adjusted temperatures
            T_b=diurnal['rural'][key][season].tas.values,
            z_b=Z_b_model['rural']
        )[0]
        Y = np.concatenate([Y,Y]) # Show two full cycles
        LINE = dict(color=colour,dash='solid')
        fig.add_trace(go.Scatter(x=X,y=Y,name=model,line=LINE))

        # To later make vertical line of T_max and T_min based on hourly data
        i_max_urban[model] = np.argmax(diurnal['urban'][key][season].tas.values)
        i_min_urban[model] = np.argmin(diurnal['urban'][key][season].tas.values)
        i_max_rural[model] = np.argmax(diurnal['rural'][key][season].tas.values)
        i_min_rural[model] = np.argmin(diurnal['rural'][key][season].tas.values)
        
        # Plot other year diurnal cycles to show inter-annual variability
        for year in years:

            diurnal_by_year_urban = (
                grouped_by_year['urban'][key][season][year]
                .groupby('time.hour')
                .mean(dim='time')
                .mean(dim='points')
            ).tas.values

            diurnal_by_year_rural = (
                grouped_by_year['rural'][key][season][year]
                .groupby('time.hour')
                .mean(dim='time')
                .mean(dim='points')
            ).tas.values

            # UHI for every year
            Y = adjust_temp(diurnal_by_year_urban,z_b = Z_b_model['urban'])[0] - adjust_temp(diurnal_by_year_rural,z_b = Z_b_model['rural'])[0]
            Y = np.concatenate([Y,Y]) # Show two full cycles
            fig.add_trace(go.Scatter(
                x=X,
                y=Y,
                name=f'{year} {model}',
                line=dict(color=colour, dash='solid', width=0.2),
                legendrank=2,
                showlegend=False
            ))

    
    # The rest is essentially just to plot the vertical lines for T_min and T_max based on the model
    # Check for common indices between models
    common_max = None
    common_min = None

    if i_max_urban['CLASS'] == i_max_urban['TEB+CLASS'] and i_max_rural['CLASS'] == i_max_rural['TEB+CLASS']:
        common_max = i_max_urban['CLASS']

    if i_min_urban['CLASS'] == i_min_urban['TEB+CLASS'] and i_min_rural['CLASS'] == i_min_rural['TEB+CLASS']:
        common_min = i_min_urban['CLASS']

    # Plot max lines
    if common_max is not None:
        fig.add_trace(go.Scatter(
        x=[X[common_max], X[common_max]],
            y=y_range,
            mode='lines',
        line=dict(color='purple', dash='dot', width=1),
            showlegend=False,
            hoverinfo='skip',
        name='Max'
        ))
        fig.add_annotation(
            x=X[common_max],
            y=y_range[1],
            text="T<sub>max</sub>",
            showarrow=False,
            font=dict(color='purple'),
            xanchor='center',
            yanchor='bottom'
        )
    else:
        for model in ['CLASS','TEB+CLASS']:
            colour = 'blue' if model == 'CLASS' else 'red'
            if i_max_urban[model] != i_max_rural[model]:
                fig.add_trace(go.Scatter(
                    x=[X[i_max_urban[model]], X[i_max_urban[model]]],
                    y=y_range,
                    mode='lines',
                    line=dict(color=colour, dash='dot', width=1),
                    showlegend=False,
                    hoverinfo='skip',
                    name=f'{model} Max Urban'
                ))
                fig.add_trace(go.Scatter(
                    x=[X[i_max_rural[model]], X[i_max_rural[model]]],
                    y=y_range,
                    mode='lines',
                    line=dict(color=colour, dash='dot', width=1),
                    showlegend=False,
                    hoverinfo='skip',
                    name=f'{model} Max Rural'
                ))
                fig.add_annotation(
                    x=X[i_max_urban[model]],
                    y=y_range[1] - offset,
                    text=f"T<sub>max urban</sub>",
                    showarrow=False,
                    font=dict(color=colour),
                    xanchor='center',
                    yanchor='bottom'
                )
                fig.add_annotation(
                    x=X[i_max_rural[model]],
                    y=y_range[0] + offset,
                    text=f"T<sub>max rural</sub>",
                    showarrow=False,
                    font=dict(color=colour),
                    xanchor='center',
                    yanchor='top'
                )
            else:
                fig.add_trace(go.Scatter(
                        x=[X[i_max_urban[model]], X[i_max_urban[model]]],
                    y=y_range,
                    mode='lines',
                    line=dict(color=colour, dash='dot', width=1),
                    showlegend=False,
                    hoverinfo='skip',
                    name=f'{model} Max'
                ))
                fig.add_annotation(
                    x=X[i_max_urban[model]],
                    y=y_range[1],
                    text=f"T<sub>max</sub>",
                    showarrow=False,
                    font=dict(color=colour),
                    xanchor='center',
                    yanchor='bottom'
                )

    # Plot min lines
    if common_min is not None:
        fig.add_trace(go.Scatter(
            x=[X[common_min], X[common_min]],
                    y=y_range,
                    mode='lines',
            line=dict(color='purple', dash='dot', width=1),
                    showlegend=False,
                    hoverinfo='skip',
            name='Min'
        ))
        fig.add_annotation(
            x=X[common_min],
            y=y_range[1],
            text="T<sub>min</sub>",
            showarrow=False,
            font=dict(color='purple'),
            xanchor='center',
            yanchor='bottom'
        )
    else:
        for model in ['CLASS','TEB+CLASS']:
            colour = 'blue' if model == 'CLASS' else 'red'
            if i_min_urban[model] != i_min_rural[model]:
                fig.add_trace(go.Scatter(
                    x=[X[i_min_urban[model]], X[i_min_urban[model]]],
                    y=y_range,
                    mode='lines',
                    line=dict(color=colour, dash='dot', width=1),
                    showlegend=False,
                    hoverinfo='skip',
                    name=f'{model} Min Urban'
                ))
                fig.add_trace(go.Scatter(
                    x=[X[i_min_rural[model]], X[i_min_rural[model]]],
                    y=y_range,
                    mode='lines',
                    line=dict(color=colour, dash='dot', width=1),
                    showlegend=False,
                    hoverinfo='skip',
                    name=f'{model} Min Rural'
                ))
                fig.add_annotation(
                    x=X[i_min_urban[model]],
                    y=y_range[1] - offset,
                    text=f"T<sub>min urban</sub>",
                    showarrow=False,
                    font=dict(color=colour),
                    xanchor='center',
                    yanchor='bottom'
                )
                fig.add_annotation(
                    x=X[i_min_rural[model]],
                    y=y_range[0] + offset,
                    text=f"T<sub>min rural</sub>",
                    showarrow=False,
                    font=dict(color=colour),
                    xanchor='center',
                    yanchor='top'
                )
            else:
                fig.add_trace(go.Scatter(
                    x=[X[i_min_urban[model]], X[i_min_urban[model]]],
                    y=y_range,
                    mode='lines',
                    line=dict(color=colour, dash='dot', width=1),
                    showlegend=False,
                    hoverinfo='skip',
                    name=f'{model} Min'
                ))
                fig.add_annotation(
                    x=X[i_min_urban[model]],
                    y=y_range[1],
                    text=f"T<sub>min</sub>",
                    showarrow=False,
                    font=dict(color=colour),
                    xanchor='center',
                    yanchor='bottom'
                )
        
    # zero line with no legend
    fig.add_trace(
        go.Scatter(
            x=[X[0], X[-1]],
            y=[0, 0],
            mode='lines',
            line=dict(color='black', dash='dash', width=1),
            showlegend=False, 
            hoverinfo='skip'  
        )
    )

    fig.update_xaxes(
        range=[0, 24],
        tickmode='array',
        tickvals=list(range(0, 25, 3)),
        ticktext=[f'{h:02d}:00' for h in range(0, 25, 3)]
    )
    fig.update_yaxes(range=y_range)

    fig.update_layout(
        title=dict(text=f'{season} Diurnal Urban Heat Island Cycle'),
        xaxis=dict(title=dict(text='Hour of day (UTC-4)')),
        yaxis=dict(title=dict(text='UHI (°C)')),
        legend=dict(title=dict(text='Model')),
    )
    fig.show()
    # fig.write_html(f'/runoff/gulley/UHI_plots/cycles/diurnal/{season}_UHI_diurnal_simobs.html')


In [135]:
# Display UHI with CI based on inter-annual variability

X = diurnal['urban']['tas_C']['JJA'].hour - 4 # Time of day converted to UTC-4
X = np.concatenate([X,X+24]) # Show two diurnal cycles in the x-axis

y_range = [-1,3.2] # range of the graph
offset = 0.4       # for annotations to not interfere with axes

for season in ['JJA','SON','DJF','MAM']:
    fig = go.Figure()

    # To connect with hottest/coldest part of the day by season/model/land cover
    i_max_urban = {} 
    i_min_urban = {}
    i_max_rural = {}
    i_min_rural = {}

    for key,model,colour in zip(['tas_C','tas_T'],['CLASS','TEB+CLASS'],['blue','red']):
        
        # UHI
        Y = adjust_temp( # urban adjusted temperatures
            T_b=diurnal['urban'][key][season].tas.values,
            z_b=Z_b_model['urban']
        )[0] - adjust_temp( # rural adjusted temperatures
            T_b=diurnal['rural'][key][season].tas.values,
            z_b=Z_b_model['rural']
        )[0]
        Y = np.concatenate([Y,Y]) # Show two full cycles
        LINE = dict(color=colour,dash='solid')
        fig.add_trace(go.Scatter(x=X,y=Y,name=model,line=LINE))

        # To later make vertical line of T_max and T_min based on hourly data
        i_max_urban[model] = np.argmax(diurnal['urban'][key][season].tas.values)
        i_min_urban[model] = np.argmin(diurnal['urban'][key][season].tas.values)
        i_max_rural[model] = np.argmax(diurnal['rural'][key][season].tas.values)
        i_min_rural[model] = np.argmin(diurnal['rural'][key][season].tas.values)

        # For painting confidence intervals onto graph
        CI = diurnal_UHI_CI[key][season].tas.values
        CI = np.concatenate([CI,CI])
        
        opacity = 0.2
        if colour == 'red':
            fill_colour = f'rgba(255, 0, 0, {opacity})' # with limited opacity
        else: # blue
            fill_colour =  f'rgba(0, 0, 255, {opacity})'

        # Lower part of confidence intervals
        fig.add_trace(go.Scatter(
            x=X,
            y=Y-CI,
            mode='lines',
            line=dict(width=0,color=colour),          
            showlegend=False,
            name=f'{model} 95% CI'
        ))

        # Upper part of confidence intervals
        fig.add_trace(go.Scatter(
            x=X,
            y=Y+CI,
            mode='lines',
            line=dict(width=0,color=colour), 
            fill='tonexty',              # fills to the trace added just before this one
            fillcolor=fill_colour,
            showlegend=True,
            name=f'{model} 95% CI'
        ))

    
    # The rest is essentially just to plot the vertical lines for T_min and T_max based on the model
    # Check for common indices between models
    common_max = None
    common_min = None

    if i_max_urban['CLASS'] == i_max_urban['TEB+CLASS'] and i_max_rural['CLASS'] == i_max_rural['TEB+CLASS']:
        common_max = i_max_urban['CLASS']

    if i_min_urban['CLASS'] == i_min_urban['TEB+CLASS'] and i_min_rural['CLASS'] == i_min_rural['TEB+CLASS']:
        common_min = i_min_urban['CLASS']

    # Plot max lines
    if common_max is not None:
        fig.add_trace(go.Scatter(
        x=[X[common_max], X[common_max]],
            y=y_range,
            mode='lines',
        line=dict(color='purple', dash='dot', width=1),
            showlegend=False,
            hoverinfo='skip',
        name='Max'
        ))
        fig.add_annotation(
            x=X[common_max],
            y=y_range[1],
            text="T<sub>max</sub>",
            showarrow=False,
            font=dict(color='purple'),
            xanchor='center',
            yanchor='bottom'
        )
    else:
        for model in ['CLASS','TEB+CLASS']:
            colour = 'blue' if model == 'CLASS' else 'red'
            if i_max_urban[model] != i_max_rural[model]:
                fig.add_trace(go.Scatter(
                    x=[X[i_max_urban[model]], X[i_max_urban[model]]],
                    y=y_range,
                    mode='lines',
                    line=dict(color=colour, dash='dot', width=1),
                    showlegend=False,
                    hoverinfo='skip',
                    name=f'{model} Max Urban'
                ))
                fig.add_trace(go.Scatter(
                    x=[X[i_max_rural[model]], X[i_max_rural[model]]],
                    y=y_range,
                    mode='lines',
                    line=dict(color=colour, dash='dot', width=1),
                    showlegend=False,
                    hoverinfo='skip',
                    name=f'{model} Max Rural'
                ))
                fig.add_annotation(
                    x=X[i_max_urban[model]],
                    y=y_range[1] - offset,
                    text=f"T<sub>max urban</sub>",
                    showarrow=False,
                    font=dict(color=colour),
                    xanchor='center',
                    yanchor='bottom'
                )
                fig.add_annotation(
                    x=X[i_max_rural[model]],
                    y=y_range[0] + offset,
                    text=f"T<sub>max rural</sub>",
                    showarrow=False,
                    font=dict(color=colour),
                    xanchor='center',
                    yanchor='top'
                )
            else:
                fig.add_trace(go.Scatter(
                        x=[X[i_max_urban[model]], X[i_max_urban[model]]],
                    y=y_range,
                    mode='lines',
                    line=dict(color=colour, dash='dot', width=1),
                    showlegend=False,
                    hoverinfo='skip',
                    name=f'{model} Max'
                ))
                fig.add_annotation(
                    x=X[i_max_urban[model]],
                    y=y_range[1],
                    text=f"T<sub>max</sub>",
                    showarrow=False,
                    font=dict(color=colour),
                    xanchor='center',
                    yanchor='bottom'
                )

    # Plot min lines
    if common_min is not None:
        fig.add_trace(go.Scatter(
            x=[X[common_min], X[common_min]],
                    y=y_range,
                    mode='lines',
            line=dict(color='purple', dash='dot', width=1),
                    showlegend=False,
                    hoverinfo='skip',
            name='Min'
        ))
        fig.add_annotation(
            x=X[common_min],
            y=y_range[1],
            text="T<sub>min</sub>",
            showarrow=False,
            font=dict(color='purple'),
            xanchor='center',
            yanchor='bottom'
        )
    else:
        for model in ['CLASS','TEB+CLASS']:
            colour = 'blue' if model == 'CLASS' else 'red'
            if i_min_urban[model] != i_min_rural[model]:
                fig.add_trace(go.Scatter(
                    x=[X[i_min_urban[model]], X[i_min_urban[model]]],
                    y=y_range,
                    mode='lines',
                    line=dict(color=colour, dash='dot', width=1),
                    showlegend=False,
                    hoverinfo='skip',
                    name=f'{model} Min Urban'
                ))
                fig.add_trace(go.Scatter(
                    x=[X[i_min_rural[model]], X[i_min_rural[model]]],
                    y=y_range,
                    mode='lines',
                    line=dict(color=colour, dash='dot', width=1),
                    showlegend=False,
                    hoverinfo='skip',
                    name=f'{model} Min Rural'
                ))
                fig.add_annotation(
                    x=X[i_min_urban[model]],
                    y=y_range[1] - offset,
                    text=f"T<sub>min urban</sub>",
                    showarrow=False,
                    font=dict(color=colour),
                    xanchor='center',
                    yanchor='bottom'
                )
                fig.add_annotation(
                    x=X[i_min_rural[model]],
                    y=y_range[0] + offset,
                    text=f"T<sub>min rural</sub>",
                    showarrow=False,
                    font=dict(color=colour),
                    xanchor='center',
                    yanchor='top'
                )
            else:
                fig.add_trace(go.Scatter(
                    x=[X[i_min_urban[model]], X[i_min_urban[model]]],
                    y=y_range,
                    mode='lines',
                    line=dict(color=colour, dash='dot', width=1),
                    showlegend=False,
                    hoverinfo='skip',
                    name=f'{model} Min'
                ))
                fig.add_annotation(
                    x=X[i_min_urban[model]],
                    y=y_range[1],
                    text=f"T<sub>min</sub>",
                    showarrow=False,
                    font=dict(color=colour),
                    xanchor='center',
                    yanchor='bottom'
                )
        
    # zero line with no legend
    fig.add_trace(
        go.Scatter(
            x=[X[0], X[-1]],
            y=[0, 0],
            mode='lines',
            line=dict(color='black', dash='dash', width=1),
            showlegend=False, 
            hoverinfo='skip'  
        )
    )

    fig.update_xaxes(
        range=[0, 24],
        tickmode='array',
        tickvals=list(range(0, 25, 3)),
        ticktext=[f'{h:02d}:00' for h in range(0, 25, 3)]
    )
    fig.update_yaxes(range=y_range)

    fig.update_layout(
        title=dict(text=f'{season} Diurnal Urban Heat Island Cycle'),
        xaxis=dict(title=dict(text='Hour of day (UTC-4)')),
        yaxis=dict(title=dict(text='UHI (°C)')),
        legend=dict(title=dict(text='Model')),
    )
    # fig.show()
    fig.write_html(f'/runoff/gulley/UHI_plots/cycles/diurnal/{season}_UHI_diurnal_simobs_CI.html')


### 2. Plot annual cycles from model and observational data
Groups and averages model & station data by month

In [162]:
# Load all daily/subdaily simobs data
series = {}
# Calculate monthly averages of simobs
monthly = {} # Overall cycle by monthly averages
monthly_by_year = {} # Average cycle of monthly averages each year
monthly_CI = {} # For confidence intervals based on inter-annual variability


for urban_or_rural,station_set in zip(['urban','rural'],[obs_urban,obs_rural]):
    series[urban_or_rural] = {}
    monthly[urban_or_rural] = {}
    monthly_by_year[urban_or_rural] = {}
    monthly_CI[urban_or_rural] = {}

    for key in ['tas_C','tas_T','tasmin_C','tasmin_T','tasmax_C','tasmax_T']:
        series[urban_or_rural][key] = xr.open_mfdataset(f'/runoff/gulley/St_Laurent/intermediates/sim/series_at_obs_locations/{key}_{urban_or_rural}.nc')
    
    # Load all daily obs data
    monthly_by_year[urban_or_rural]['tasmin_S'] = station_set['tasmin'].groupby(['time.year', 'time.month']).mean(dim='time').mean(dim='station') + 273.15
    monthly_by_year[urban_or_rural]['tasmax_S'] = station_set['tasmax'].groupby(['time.year', 'time.month']).mean(dim='time').mean(dim='station') + 273.15
    monthly[urban_or_rural]['tasmin_S'] = monthly_by_year[urban_or_rural]['tasmin_S'].mean(dim='year')
    monthly[urban_or_rural]['tasmax_S'] = monthly_by_year[urban_or_rural]['tasmax_S'].mean(dim='year')
    monthly_CI[urban_or_rural]['tasmin_S'] = (monthly_by_year[urban_or_rural]['tasmin_S'].std(dim='year')/ np.sqrt(n)) * stats.t.ppf(1 - alpha/2, n-1)
    monthly_CI[urban_or_rural]['tasmax_S'] = (monthly_by_year[urban_or_rural]['tasmax_S'].std(dim='year')/ np.sqrt(n)) * stats.t.ppf(1 - alpha/2, n-1)

    for f,key in zip(['tasmin','tasmin','tasmax','tasmax'],['tasmin_C','tasmin_T','tasmax_C','tasmax_T']): # Can exclude tas because it's just the average of tasmax and tasmin
        monthly_by_year[urban_or_rural][key] = series[urban_or_rural][key].groupby(['time.year', 'time.month']).mean(dim='time').mean(dim='points')[f] # average for each year (12 months x 23 years)
        monthly[urban_or_rural][key] = monthly_by_year[urban_or_rural][key].mean(dim='year') # average over all years (12 months)
        monthly_CI[urban_or_rural][key] = (monthly_by_year[urban_or_rural][key].std(dim='year')/np.sqrt(n)) * stats.t.ppf(1-alpha/2,n-1)

In [163]:
# Scaling temperatures to some common altitude based on DABL assumptions
Z_b_obs = {}
Z_b_obs['urban'] = np.mean(obs_urban.elev.values)
Z_b_obs['rural'] = np.mean(obs_rural.elev.values)
print(f'orog (m) before altitude scaling for each station set:\nmodel values {Z_b_model}\nobservation values {Z_b_obs}\nscaled to {Z_a}m')

orog (m) before altitude scaling for each station set:
model values {'urban': 58.576568175475465, 'rural': 46.045161577295985}
observation values {'urban': 54.5, 'rural': 54.142857142857146}
scaled to 54.5m


In [ ]:
# Create confidence intervals for the monthly-averaged UHI based on inter-annual variability
monthly_UHI_CI = {}
for key, Z_b in zip(['tasmin_C','tasmax_C','tasmin_T','tasmax_T','tasmin_S','tasmax_S'],[Z_b_model,Z_b_model,Z_b_model,Z_b_model,Z_b_obs,Z_b_obs]):
    urban_adjusted = adjust_temp(monthly_by_year['urban'][key].values, z_b = Z_b['urban'])
    rural_adjusted = adjust_temp(monthly_by_year['rural'][key].values, z_b = Z_b['rural'])
    
    monthly_UHI_CI[key] = ((monthly_by_year['urban'][key].copy(data=urban_adjusted) - monthly_by_year['rural'][key].copy(data=rural_adjusted)).std(dim='year') /np.sqrt(n)) * stats.t.ppf(1-alpha/2,n-1)

tasmin_C
tasmax_C
tasmin_T
tasmax_T
tasmin_S
tasmax_S


In [171]:
urban_adjusted_da = monthly_by_year['urban'][key].copy(data=urban_adjusted)
rural_adjusted_da = monthly_by_year['rural'][key].copy(data=rural_adjusted)

UHI_by_year = urban_adjusted_da - rural_adjusted_da  # shape: (year, month)

# Print the std across years for each month
print(UHI_by_year.std(dim='year', ddof=1))

<xarray.DataArray 'tasmax' (month: 12)> Size: 96B
array([0.46619264, 0.38154082, 0.29084371, 0.32345545, 0.23517788,
       0.462747  , 0.24575582, 0.28223132, 0.24635678, 0.19440296,
       0.33476871, 0.29122101])
Coordinates:
  * month    (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12


In [ ]:
# Monthly temperatures by model and for each year
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
X = monthly['urban']['tasmin_C'].month.values

for f,field_name in zip(['tasmin','tasmax'],['Minimum Daily Temperature','Maximum Daily Temperature']):
    fig = go.Figure()
    for urban_or_rural,linestyle in zip(['urban','rural'],['solid','dash']):
        for key,model,colour, Z_b in zip([f'{f}_C',f'{f}_T',f'{f}_S'],['CLASS','TEB+CLASS','Observation'],['blue','red','black'], [Z_b_model,Z_b_model,Z_b_obs]):

            # Adding the average monthly temperature line
            Y = adjust_temp(monthly[urban_or_rural][key].values, z_b = Z_b[urban_or_rural])[0] - 273.15
            LINE = dict(color=colour,dash=linestyle,width=3)
            fig.add_trace(go.Scatter(x=X,y=Y,name=f'{model} {urban_or_rural}',line=LINE,legendrank=2))
    
    # Adding the inter-annual lines
    for year in years:
        for urban_or_rural,linestyle in zip(['urban','rural'],['solid','dash']):
            for key,model,colour, Z_b in zip([f'{f}_C',f'{f}_T',f'{f}_S'],['CLASS','TEB+CLASS','Observation'],['blue','red','black'], [Z_b_model,Z_b_model,Z_b_obs]):
                Y = adjust_temp(monthly_by_year[urban_or_rural][key].sel(year=year).values,z_b = Z_b[urban_or_rural])[0] - 273.15
                # Adding the average monthly temperature lines of each year
                fig.add_trace(go.Scatter(
                        x=X,
                        y=Y,
                        name=f'{year} {model} {urban_or_rural}',
                        line=dict(color=colour, dash=linestyle, width=0.3),
                        legendrank=2,
                        showlegend=True,
                        mode='lines'
                ))

    fig.update_layout(
        title=dict(text=f'Monthly Averages of {field_name}'),
        xaxis=dict(title=dict(text='Month')),
        yaxis=dict(title=dict(text='Temperature (°C)')),
        legend=dict(title=dict(text='Model and Station Type')),
    )
    fig.update_xaxes(
        tickmode='array',
        tickvals=list(range(1, 13)), 
        ticktext=months,
        range=[1, 12]              
    )
    fig.show()

    # fig.write_html(f'/runoff/gulley/UHI_plots/cycles/annual/{f}_monthly_simobs.html')

In [156]:
# Monthly temperatures by model and 95% CI based on inter-annual variability  
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
X = monthly['urban']['tasmin_C'].month.values

for f,field_name in zip(['tasmin','tasmax'],['Minimum Daily Temperature','Maximum Daily Temperature']):
    fig = go.Figure()
    for urban_or_rural,linestyle in zip(['urban','rural'],['solid','dash']):
        for key,model,colour, Z_b in zip([f'{f}_C',f'{f}_T',f'{f}_S'],['CLASS','TEB+CLASS','Observation'],['blue','red','black'], [Z_b_model,Z_b_model,Z_b_obs]):

            # Adding the average monthly temperature line
            Y = adjust_temp(monthly[urban_or_rural][key].values, z_b = Z_b[urban_or_rural])[0] - 273.15
            LINE = dict(color=colour,dash=linestyle,width=3)
            fig.add_trace(go.Scatter(x=X,y=Y,name=f'{model} {urban_or_rural}',line=LINE,legendrank=2))
    
            # For painting confidence intervals onto graph
            CI = monthly_CI[urban_or_rural][key].values
            opacity = 0.2
            if urban_or_rural == 'rural':
                opacity = 0.1

            if colour == 'red':
                fill_colour = f'rgba(255, 0, 0, {opacity})' # with limited opacity
            elif colour == 'blue':
                fill_colour =  f'rgba(0, 0, 255, {opacity})'
            else : # black
                fill_colour = f'rgba(0, 0, 0, {opacity})'
            
            if urban_or_rural == 'rural':
                fillpattern = dict(
                    shape='/',          # hatch style
                    size=8,             # spacing between lines
                    solidity=0.2,       # density of the pattern (0–1)
                    fgcolor=fill_colour,     # hatch line color
                )
            else:
                fillpattern = None

            # Lower part of confidence intervals
            fig.add_trace(go.Scatter(
                x=X,
                y=Y-CI,
                mode='lines',
                line=dict(width=0,color=colour),          
                showlegend=False,
                name=f'{model} {urban_or_rural} 95% CI',
                legendrank=1
            ))

            # Upper part of confidence intervals
            fig.add_trace(go.Scatter(
                x=X,
                y=Y+CI,
                mode='lines',
                line=dict(width=0,color=colour), 
                fill='tonexty',              # fills to the trace added just before this one
                fillcolor=fill_colour,
                showlegend=True,
                fillpattern=fillpattern,
                name=f'{model} {urban_or_rural} 95% CI',
                legendrank=1
            ))

    fig.update_layout(
        title=dict(text=f'Monthly Averages of {field_name}'),
        xaxis=dict(title=dict(text='Month')),
        yaxis=dict(title=dict(text='Temperature (°C)')),
        legend=dict(title=dict(text='Model and Station Type')),
    )
    fig.update_xaxes(
        tickmode='array',
        tickvals=list(range(1, 13)), 
        ticktext=months,
        range=[1, 12]              
    )
    # fig.show()

    fig.write_html(f'/runoff/gulley/UHI_plots/cycles/annual/{f}_monthly_simobs_CI.html')

In [110]:
# Display UHI evolution for every year
y_range=[-2.5,4]
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
X = monthly['urban']['tasmin_C'].month.values

for f,field_name in zip(['tasmin','tasmax'],['Minimum Daily Temperature','Maximum Daily Temperature']):
    fig = go.Figure()
    for key,model,colour, Z_b in zip([f'{f}_C',f'{f}_T',f'{f}_S'],['CLASS','TEB+CLASS','Observation'],['blue','red','black'],[Z_b_model,Z_b_model,Z_b_obs]):
        # for point,name in zip(monthly[key].points.values, station_set.station_name.values):
        #     fig.add_trace(go.Scatter(x=monthly[key].sel(points=point).month.values,y=monthly[key].sel(points=point)[f].values - 273.15,name=f'{name} ({model})'))
        

        Y = adjust_temp(monthly['urban'][key].values,z_b = Z_b['urban'])[0] - adjust_temp(monthly['rural'][key].values,z_b = Z_b['rural'])[0]
            
        LINE = dict(color=colour,dash='solid')
        fig.add_trace(go.Scatter(x=X,y=Y,name=f'{model}',line=LINE))
    # zero line with no legend
    fig.add_trace(
        go.Scatter(
            x=[X[0], X[-1]],  # full range of x data
            y=[0, 0],
            mode='lines',
            line=dict(color='black', dash='dash', width=1),
            showlegend=False, 
            hoverinfo='skip',
            legendrank=1  
        )
    )
    
    # Adding the inter-annual lines
    for year in years:
        for key,model,colour, Z_b in zip([f'{f}_C',f'{f}_T',f'{f}_S'],['CLASS','TEB+CLASS','Observation'],['blue','red','black'], [Z_b_model,Z_b_model,Z_b_obs]):
            Y = adjust_temp(monthly_by_year['urban'][key].sel(year=year).values,z_b = Z_b['urban'])[0] - adjust_temp(monthly_by_year['rural'][key].sel(year=year).values,z_b = Z_b['rural'])[0]
            
            # Adding the average monthly temperature lines of each year
            fig.add_trace(go.Scatter(
                    x=X,
                    y=Y,
                    name=f'{year} {model} {urban_or_rural}',
                    line=dict(color=colour, dash='solid', width=0.3),
                    legendrank=2,
                    showlegend=False,
                    mode='lines'
            ))


    fig.update_layout(
        title=dict(text=f'Monthly Averages of UHI for {field_name}'),
        xaxis=dict(title=dict(text='Month')),
        yaxis=dict(title=dict(text='UHI (°C)')),
        legend=dict(title=dict(text='Model or Observation')),
    )
    fig.update_xaxes(
        tickmode='array',
        tickvals=list(range(1, 13)), 
        ticktext=months,
        range=[1, 12]              
    )
    fig.update_yaxes(range=y_range) # set extent of y axis
    fig.show()

    # fig.write_html(f'/runoff/gulley/UHI_plots/cycles/annual/{f}_UHI_monthly_simobs.html')

In [172]:
# Display UHI with CI based on inter-annual variability 
y_range=[-2.5,4]
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
X = monthly['urban']['tasmin_C'].month.values

for f,field_name in zip(['tasmin','tasmax'],['Minimum Daily Temperature','Maximum Daily Temperature']):
    fig = go.Figure()
    for key,model,colour, Z_b in zip([f'{f}_C',f'{f}_T',f'{f}_S'],['CLASS','TEB+CLASS','Observation'],['blue','red','black'],[Z_b_model,Z_b_model,Z_b_obs]):
        # for point,name in zip(monthly[key].points.values, station_set.station_name.values):
        #     fig.add_trace(go.Scatter(x=monthly[key].sel(points=point).month.values,y=monthly[key].sel(points=point)[f].values - 273.15,name=f'{name} ({model})'))
        

        Y = adjust_temp(monthly['urban'][key].values,z_b = Z_b['urban'])[0] - adjust_temp(monthly['rural'][key].values,z_b = Z_b['rural'])[0]
            
        LINE = dict(color=colour,dash='solid')
        fig.add_trace(go.Scatter(x=X,y=Y,name=f'{model}',line=LINE))

        # For painting confidence intervals onto graph
        CI = monthly_UHI_CI[key].values
        opacity = 0.2

        if colour == 'red':
            fill_colour = f'rgba(255, 0, 0, {opacity})' # with limited opacity
        elif colour == 'blue':
            fill_colour =  f'rgba(0, 0, 255, {opacity})'
        else : # black
            fill_colour = f'rgba(0, 0, 0, {opacity})'

        # Lower part of confidence intervals
        fig.add_trace(go.Scatter(
            x=X,
            y=Y-CI,
            mode='lines',
            line=dict(width=0,color=colour),          
            showlegend=False,
            name=f'{model} 95% CI',
            legendrank=1
        ))

        # Upper part of confidence intervals
        fig.add_trace(go.Scatter(
            x=X,
            y=Y+CI,
            mode='lines',
            line=dict(width=0,color=colour), 
            fill='tonexty',              # fills to the trace added just before this one
            fillcolor=fill_colour,
            showlegend=True,
            name=f'{model} 95% CI',
            legendrank=1
        ))

    # zero line with no legend
    fig.add_trace(
        go.Scatter(
            x=[X[0], X[-1]],  # full range of x data
            y=[0, 0],
            mode='lines',
            line=dict(color='black', dash='dash', width=1),
            showlegend=False, 
            hoverinfo='skip',
            legendrank=1  
        )
    )

    fig.update_layout(
        title=dict(text=f'Monthly Averages of UHI for {field_name}'),
        xaxis=dict(title=dict(text='Month')),
        yaxis=dict(title=dict(text='UHI (°C)')),
        legend=dict(title=dict(text='Model or Observation')),
    )
    fig.update_xaxes(
        tickmode='array',
        tickvals=list(range(1, 13)), 
        ticktext=months,
        range=[1, 12]              
    )
    fig.update_yaxes(range=y_range) # set extent of y axis
    # fig.show()

    fig.write_html(f'/runoff/gulley/UHI_plots/cycles/annual/{f}_UHI_monthly_simobs_CI.html')